<a href="https://colab.research.google.com/github/kumarsirish/FDP-AGENENTIC-AI-RAG/blob/main/rag-agentic-01/fictional-depatment-rag-agentic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Agentic RAG

This notebook walks through building an Agentic RAG system using LangChain and FAISS.

**Steps:**
1. Install dependencies
2. Ingest documents into a FAISS vector store
3. Build a LangChain agent with a document search tool
4. Ask questions

## 1. Install Dependencies

In [1]:
import os

if not os.path.exists('requirements.txt'):
    ! wget https://raw.githubusercontent.com/kumarsirish/FDP-AGENENTIC-AI-RAG/main/rag-agentic-01/requirements.txt
else:
    print('requirements.txt already exists.')
%pip install -r  requirements.txt -q


--2026-06-07 11:07:59--  https://raw.githubusercontent.com/kumarsirish/FDP-AGENENTIC-AI-RAG/main/rag-agentic-01/requirements.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 201 [text/plain]
Saving to: ‘requirements.txt’

requirements.txt    100%[===================>]     201  --.-KB/s    in 0s      

2026-06-07 11:07:59 (12.5 MB/s) - ‘requirements.txt’ saved [201/201]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 97.7 MB/s eta 0:00:00


## 2. Set API Keys

In [2]:
import os
from dotenv import load_dotenv
#from kaggle_secrets import UserSecretsClient

# Fallback: try Google Colab secrets
try:
    from google.colab import userdata
    if not os.getenv("HF_TOKEN"):
        HF_TOKEN = userdata.get("HF_TOKEN").strip()
    if not os.getenv("GEMINI_API_KEY"):
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY").strip()
except ImportError:
  pass


# Load from .env file (local development)
# load_dotenv("/home/sirkumar/FDP-AGENENTIC-AI-RAG/.env")


#Kaggle
#user_secrets = UserSecretsClient()
#HF_TOKEN = user_secrets.get_secret("HF_TOKEN").strip()
#GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY").strip()

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

#HF_TOKEN = (os.getenv("HF_TOKEN") or "").strip()
#GEMINI_API_KEY = (os.getenv("GEMINI_API_KEY") or "").strip()


if not GEMINI_API_KEY or not HF_TOKEN:
    print("Warning: HF_TOKEN or GEMINI_API_KEY not set. Please set it in .env file or Colab secrets.")

In [3]:
department_info = [
    # Department overview
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.",
    "Students can choose from 5 courses ranging from core subjects to electives and hands-on project work, with strong industry exposure.",
    "DQE offers a postgraduate module 'Foundations of Quantum AI' and an undergraduate elective 'Quantum Machine Learning' using IBM Qiskit and PennyLane.",
    "All students have access to cloud quantum hardware through IBM Quantum Network and Amazon Braket as part of their coursework.",

    # Quantum AI research
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "The Quantum AI Lab investigates Quantum Neural Networks (QNNs) using parameterized quantum circuits (PQCs) and collaborates with a national quantum computing centre on 20-qubit and 50-qubit processors.",
    "A key research thread is Quantum Reinforcement Learning (QRL), where quantum agents learn policies faster than classical counterparts on specific problem classes.",
    "Project QuLearn is DQE's flagship initiative building hybrid classical-quantum models for drug discovery and materials science.",

]


## 3. Ingest Documents into FAISS Vector Store

Defines knowledge inline, splits it into chunks, embeds them, and saves the vector store locally.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

documents = [Document(page_content=text) for text in department_info]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
docs = splitter.split_documents(documents)

# Force anonymous download for public embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2"
)

db = FAISS.from_documents(docs, embeddings)
db.save_local("vectorstore")

print(f"Vector DB created successfully ({len(docs)} chunks)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector DB created successfully (8 chunks)


In [5]:
# Create retriever — returns top 2 most similar documents for any query
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 2})
print("Retriever ready.")

Retriever ready.


## 4. Build the RAG Agent

Loads the vector store and wires it up as a tool for a LangChain ZERO_SHOT_REACT agent.

In [14]:
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_react_agent, AgentExecutor # Updated import
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # Added MessagesPlaceholder


# ── 1. Load vector store ──────────────────────────────────────────────

db = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 2})

# ── 2. Tool ───────────────────────────────────────────────────────────
@tool
def search_answers(query: str) -> str:
    """Search the knowledge base for information about the DQE department."""
    results = retriever.invoke(query)
    docs = "\n".join([doc.page_content for doc in results])
    print(f"\n[TOOL CALLED] query='{query}'")
    print(f"[RETRIEVED]:\n{docs}\n")
    return docs if docs.strip() else "No relevant documents found."



llm_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    provider="auto",           # ← KEY FIX: routes to Nebius/Fireworks/etc.
    huggingfacehub_api_token=os.getenv("HF_TOKEN"), # Use os.getenv for HF_TOKEN
    task="text-generation",
    max_new_tokens=512,
    temperature=0.1,
)


# Wrap in ChatHuggingFace — this enables bind_tools() for agentic RAG
chat_llm = ChatHuggingFace(llm=llm_endpoint, verbose=True)


# ── 4. Agent ──────────────────────────────────────────────────────────
# Define the prompt for the agent
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant for the Department of Quantum Engineering (DQE). "
     "You answer questions using the knowledge base via the search_answers tool. "
     "\n\nRules:"
     "\n- Always use search_answers when the question is about DQE — its people, courses, research, or policies."
     "\n- Do NOT use the tool for general knowledge questions (e.g., 'what is quantum computing?')."
     "\n- If the tool returns no results, say you don't have that information in the knowledge base."
     "\n- Answer concisely based only on what the tool returns. Do not hallucinate details.\n"
     "Use the following format:\n"
     "Question: the input question you must answer\n"
     "Thought: you should always think about what to do\n"
     "Action: the action to take, should be one of [{tool_names}]\n"
     "Action Input: the input to the action\n"
     "Observation: the result of the action\n"
     "... (this Thought/Action/Action Input/Observation can repeat N times)\n"
     "Thought: I now know the final answer\n"
     "Final Answer: the final answer to the original input question"),
    MessagesPlaceholder("chat_history"), # Use MessagesPlaceholder conversational memory across multiple turns
    ("human", "{input}\n{agent_scratchpad}") # Combine human input and scratchpad (which contains all previous thoughts, actions, and observations as a string
])


# Create the ReAct agent
agent = create_react_agent(chat_llm, [search_answers], prompt)

# Create the AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=[search_answers], verbose=True)

print(f"Agent ready ✅")

Agent ready ✅


## 5. Ask Questions

Edit `question` below and re-run this cell to query the agent.

In [15]:
def ask(question: str) -> str:
    response = agent_executor.invoke({"input": question, "chat_history": []})
    # The final answer from AgentExecutor is in the 'output' key
    return response["output"]

print("Answer: ",ask("What is DQE?"))



> Entering new AgentExecutor chain...
Thought: I should use the search_answers tool to find information about DQE.
Action: search_answers
Action Input: "What is DQE?"
Observ
[TOOL CALLED] query='What is DQE?"
Observ'
[RETRIEVED]:
DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.
The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.

DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.
The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.I now know the final answer
Final Answer: DQE is the Department of Quantum Engineering, which primarily focuses on Quantum AI — the intersection of quantum computing and artificial intelligence. The department has around 140 students and 20 pro

In [16]:
print("Answer: ",ask("How many students does DQE have?"))



> Entering new AgentExecutor chain...
Thought: I should use the search_answers tool to find this information.
Action: search_answers
Action Input: "How many students does DQE have?"
Observ
[TOOL CALLED] query='How many students does DQE have?"
Observ'
[RETRIEVED]:
The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.
DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.

The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.
DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.I now know the final answer
Final Answer: DQE has around 140 students.

> Finished chain.
Answer:  DQE has around 140 students.


In [17]:
print("Answer: ",ask("Whats the objective?"))



> Entering new AgentExecutor chain...
Thought: I should use the search_answers tool to find information about the objective of the Department of Quantum Engineering (DQE).

Action: search_answers
Action Input: "What is the objective of the Department of Quantum Engineering?"
Observ
[TOOL CALLED] query='What is the objective of the Department of Quantum Engineering?"
Observ'
[RETRIEVED]:
The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.
DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.

The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.
DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.I now know the final answer
Final Answer: The objective of the Department of Quantum